# 🚀 nano-vLLM 环境设置和快速开始

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zysno1/nano-vllm-learning/blob/main/notebooks/00_环境设置和快速开始.ipynb)

欢迎来到 nano-vLLM 的学习之旅！这个 Notebook 将帮助您：

- 🔧 设置 Colab 环境
- 📦 安装必要的依赖
- 🎯 运行第一个推理示例
- 🧠 理解核心概念

## 📚 学习目标

通过这个 Notebook，您将：
1. 验证环境配置是否正确
2. 理解 vLLM 的基本工作流程
3. 掌握核心概念：Tokenization、Batching、Sampling

## 🔧 环境设置

首先，让我们设置 Colab 环境并安装必要的依赖。

In [ ]:
# 检查是否在 Colab 环境中
try:
    import google.colab
    IN_COLAB = True
    print("✅ 运行在 Google Colab 环境中")
except ImportError:
    IN_COLAB = False
    print("ℹ️ 运行在本地环境中")

# 安装依赖
if IN_COLAB:
    print("📦 安装 nano-vLLM 依赖...")
    !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    !pip install transformers accelerate sentencepiece
    !pip install matplotlib seaborn numpy pandas
    
    # 克隆项目代码
    !git clone https://github.com/zysno1/nano-vllm-learning.git
    %cd nano-vllm-learning
    
    print("✅ 依赖安装完成！")
else:
    print("ℹ️ 本地环境，请确保已安装必要依赖")

## 🔍 环境检查

让我们验证环境是否正确配置。

In [ ]:
#!/usr/bin/env python3
"""
环境检查脚本
验证 nano-vLLM 运行环境是否正确配置
"""

import sys
import platform
import subprocess
from typing import Dict, List, Tuple

def check_python_version() -> Tuple[bool, str]:
    """检查 Python 版本"""
    version = sys.version_info
    if version.major == 3 and version.minor >= 8:
        return True, f"✅ Python {version.major}.{version.minor}.{version.micro}"
    else:
        return False, f"❌ Python {version.major}.{version.minor}.{version.micro} (需要 >= 3.8)"

def check_required_packages() -> List[Tuple[str, bool, str]]:
    """检查必需的 Python 包"""
    required_packages = [
        'torch',
        'transformers', 
        'numpy',
        'matplotlib'
    ]
    
    results = []
    for package in required_packages:
        try:
            __import__(package)
            # 获取版本信息
            if package == 'torch':
                import torch
                version = torch.__version__
            elif package == 'transformers':
                import transformers
                version = transformers.__version__
            elif package == 'numpy':
                import numpy
                version = numpy.__version__
            elif package == 'matplotlib':
                import matplotlib
                version = matplotlib.__version__
            else:
                version = "已安装"
            
            results.append((package, True, f"✅ {package} {version}"))
        except ImportError:
            results.append((package, False, f"❌ {package} 未安装"))
    
    return results

def check_gpu_availability() -> Tuple[bool, str]:
    """检查 GPU 可用性"""
    try:
        import torch
        if torch.cuda.is_available():
            gpu_count = torch.cuda.device_count()
            gpu_name = torch.cuda.get_device_name(0)
            return True, f"✅ GPU 可用: {gpu_name} (共 {gpu_count} 个设备)"
        else:
            return False, "⚠️ GPU 不可用，将使用 CPU (性能较慢)"
    except Exception as e:
        return False, f"❌ GPU 检查失败: {e}"

def check_memory() -> str:
    """检查内存信息"""
    try:
        import psutil
        memory = psutil.virtual_memory()
        total_gb = memory.total / (1024**3)
        available_gb = memory.available / (1024**3)
        return f"💾 内存: {available_gb:.1f}GB 可用 / {total_gb:.1f}GB 总计"
    except ImportError:
        return "💾 内存信息不可用 (psutil 未安装)"

def main():
    """主检查函数"""
    print("🔍 nano-vLLM 环境检查")
    print("=" * 50)
    
    # 系统信息
    print(f"🖥️ 系统: {platform.system()} {platform.release()}")
    print(f"🏗️ 架构: {platform.machine()}")
    
    # Python 版本检查
    python_ok, python_msg = check_python_version()
    print(python_msg)
    
    # 包检查
    print("\n📦 依赖包检查:")
    package_results = check_required_packages()
    all_packages_ok = True
    for package, ok, msg in package_results:
        print(f"  {msg}")
        if not ok:
            all_packages_ok = False
    
    # GPU 检查
    gpu_ok, gpu_msg = check_gpu_availability()
    print(f"\n{gpu_msg}")
    
    # 内存检查
    memory_msg = check_memory()
    print(memory_msg)
    
    # 总结
    print("\n" + "=" * 50)
    if python_ok and all_packages_ok:
        print("🎉 环境检查通过！可以开始学习 nano-vLLM")
        if gpu_ok:
            print("⚡ GPU 可用，推理速度会很快")
        else:
            print("⚠️ 建议使用 GPU 以获得更好的性能")
    else:
        print("❌ 环境配置有问题，请安装缺失的依赖")
        print("💡 在 Colab 中运行上面的安装命令")

# 运行环境检查
main()

## 🎯 第一个推理示例

现在让我们运行第一个 nano-vLLM 推理示例！

In [ ]:
#!/usr/bin/env python3
"""
Hello vLLM - 第一个推理示例
展示 nano-vLLM 的基本使用方法
"""

import time
from typing import List, Dict, Any

# 模拟 nano-vLLM 的核心组件
class SimpleTokenizer:
    """简化的分词器"""
    
    def __init__(self):
        # 简单的词汇表 (实际应用中会使用 HuggingFace tokenizer)
        self.vocab = {
            "<pad>": 0, "<unk>": 1, "<s>": 2, "</s>": 3,
            "hello": 4, "world": 5, "how": 6, "are": 7, "you": 8,
            "I": 9, "am": 10, "fine": 11, "thank": 12, "thanks": 13,
            "good": 14, "great": 15, "awesome": 16, "!": 17, "?": 18, ".": 19
        }
        self.reverse_vocab = {v: k for k, v in self.vocab.items()}
    
    def encode(self, text: str) -> List[int]:
        """编码文本为 token IDs"""
        tokens = text.lower().replace("!", " !").replace("?", " ?").replace(".", " .").split()
        return [self.vocab.get(token, self.vocab["<unk>"]) for token in tokens]
    
    def decode(self, token_ids: List[int]) -> str:
        """解码 token IDs 为文本"""
        tokens = [self.reverse_vocab.get(id, "<unk>") for id in token_ids]
        return " ".join(tokens).replace(" !", "!").replace(" ?", "?").replace(" .", ".")

class SimpleModel:
    """简化的语言模型"""
    
    def __init__(self, tokenizer: SimpleTokenizer):
        self.tokenizer = tokenizer
        # 简单的响应模板 (实际应用中会使用神经网络)
        self.responses = {
            "hello": ["hello", "!", "how", "are", "you", "?"],
            "how are you": ["I", "am", "fine", "thanks", "!"],
            "thank": ["you", "are", "welcome", "!"],
            "thanks": ["you", "are", "welcome", "!"]
        }
    
    def generate(self, input_text: str, max_tokens: int = 10) -> str:
        """生成响应"""
        input_lower = input_text.lower().strip()
        
        # 查找匹配的响应
        for key, response_tokens in self.responses.items():
            if key in input_lower:
                # 编码和解码过程 (展示 tokenization)
                token_ids = [self.tokenizer.vocab.get(token, 1) for token in response_tokens]
                return self.tokenizer.decode(token_ids[:max_tokens])
        
        # 默认响应
        return "I understand. Can you tell me more?"

class NanoVLLM:
    """nano-vLLM 主类"""
    
    def __init__(self, model_name: str = "simple-chat"):
        print(f"🚀 初始化 nano-vLLM (模型: {model_name})")
        
        # 初始化组件
        self.tokenizer = SimpleTokenizer()
        self.model = SimpleModel(self.tokenizer)
        self.model_name = model_name
        
        print("✅ 模型加载完成")
    
    def generate(self, 
                prompt: str, 
                max_tokens: int = 50,
                temperature: float = 0.7,
                top_p: float = 0.9) -> Dict[str, Any]:
        """生成文本"""
        
        start_time = time.time()
        
        print(f"📝 输入: {prompt}")
        print(f"⚙️ 参数: max_tokens={max_tokens}, temperature={temperature}, top_p={top_p}")
        
        # 1. Tokenization
        print("\n🔤 步骤 1: Tokenization")
        input_tokens = self.tokenizer.encode(prompt)
        print(f"   输入 tokens: {input_tokens}")
        print(f"   Token 数量: {len(input_tokens)}")
        
        # 2. 模型推理
        print("\n🧠 步骤 2: 模型推理")
        print("   正在生成响应...")
        time.sleep(0.5)  # 模拟推理时间
        
        output_text = self.model.generate(prompt, max_tokens)
        
        # 3. 后处理
        print("\n🔧 步骤 3: 后处理")
        output_tokens = self.tokenizer.encode(output_text)
        print(f"   输出 tokens: {output_tokens}")
        print(f"   生成 token 数量: {len(output_tokens)}")
        
        end_time = time.time()
        inference_time = end_time - start_time
        
        result = {
            "prompt": prompt,
            "generated_text": output_text,
            "input_tokens": len(input_tokens),
            "output_tokens": len(output_tokens),
            "inference_time": inference_time,
            "tokens_per_second": len(output_tokens) / inference_time if inference_time > 0 else 0
        }
        
        return result

def demonstrate_basic_usage():
    """演示基本使用方法"""
    print("🎯 nano-vLLM 基本使用演示")
    print("=" * 60)
    
    # 初始化 nano-vLLM
    llm = NanoVLLM("simple-chat")
    
    # 测试用例
    test_prompts = [
        "Hello!",
        "How are you?",
        "Thank you for your help"
    ]
    
    results = []
    
    for i, prompt in enumerate(test_prompts, 1):
        print(f"\n📋 测试 {i}/{len(test_prompts)}")
        print("-" * 40)
        
        result = llm.generate(
            prompt=prompt,
            max_tokens=20,
            temperature=0.7
        )
        
        print(f"\n✨ 结果: {result['generated_text']}")
        print(f"⏱️ 推理时间: {result['inference_time']:.3f}s")
        print(f"🚀 速度: {result['tokens_per_second']:.1f} tokens/s")
        
        results.append(result)
    
    return results

def show_performance_summary(results: List[Dict[str, Any]]):
    """显示性能总结"""
    print("\n" + "=" * 60)
    print("📊 性能总结")
    print("=" * 60)
    
    total_input_tokens = sum(r['input_tokens'] for r in results)
    total_output_tokens = sum(r['output_tokens'] for r in results)
    total_time = sum(r['inference_time'] for r in results)
    avg_speed = sum(r['tokens_per_second'] for r in results) / len(results)
    
    print(f"📝 总输入 tokens: {total_input_tokens}")
    print(f"📤 总输出 tokens: {total_output_tokens}")
    print(f"⏱️ 总推理时间: {total_time:.3f}s")
    print(f"🚀 平均速度: {avg_speed:.1f} tokens/s")
    
    print("\n💡 关键概念理解:")
    print("1. Tokenization: 将文本转换为模型可理解的数字")
    print("2. 推理: 模型根据输入生成输出的过程")
    print("3. 解码: 将模型输出转换回人类可读的文本")
    print("4. 批处理: 同时处理多个请求以提高效率")

# 运行演示
print("🎉 欢迎使用 nano-vLLM！")
print("这是一个简化的演示，帮助您理解核心概念\n")

results = demonstrate_basic_usage()
show_performance_summary(results)

print("\n🎓 恭喜！您已经完成了第一个 nano-vLLM 示例")
print("📚 接下来可以学习更高级的概念和技术")

## 🧠 核心概念理解

让我们深入理解 nano-vLLM 的核心概念。

In [ ]:
#!/usr/bin/env python3
"""
核心概念演示
深入理解 Tokenization、Batching、Sampling 等核心概念
"""

import numpy as np
import matplotlib.pyplot as plt
import time
from typing import List, Dict, Any, Tuple
import random

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

class ConceptDemonstrator:
    """核心概念演示器"""
    
    def __init__(self):
        self.vocab_size = 1000
        self.setup_demo_data()
    
    def setup_demo_data(self):
        """设置演示数据"""
        # 模拟词汇表
        self.sample_tokens = [
            "hello", "world", "how", "are", "you", "I", "am", "fine", 
            "thank", "thanks", "good", "great", "awesome", "!", "?", "."
        ]
        
        # 模拟文本示例
        self.sample_texts = [
            "Hello, how are you?",
            "I am doing great, thanks!",
            "What is your favorite color?",
            "The weather is awesome today."
        ]
    
    def demonstrate_tokenization(self):
        """演示 Tokenization 过程"""
        print("🔤 概念 1: Tokenization (分词)")
        print("=" * 50)
        print("Tokenization 是将人类语言转换为模型可理解的数字序列的过程")
        print()
        
        for i, text in enumerate(self.sample_texts[:2], 1):
            print(f"示例 {i}: {text}")
            
            # 简单分词 (实际中会更复杂)
            tokens = text.lower().replace("?", " ?").replace("!", " !").replace(".", " .").split()
            token_ids = [hash(token) % 100 for token in tokens]  # 模拟 token ID
            
            print(f"  Tokens: {tokens}")
            print(f"  Token IDs: {token_ids}")
            print(f"  序列长度: {len(tokens)}")
            print()
        
        # 可视化 tokenization
        self.visualize_tokenization()
    
    def visualize_tokenization(self):
        """可视化 tokenization 过程"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        
        # 文本长度分布
        text_lengths = [len(text.split()) for text in self.sample_texts]
        ax1.bar(range(len(text_lengths)), text_lengths, color='skyblue', alpha=0.7)
        ax1.set_title('Text Length Distribution')
        ax1.set_xlabel('Sample Index')
        ax1.set_ylabel('Number of Tokens')
        ax1.grid(True, alpha=0.3)
        
        # Token 频率分布 (模拟)
        token_freq = np.random.zipf(1.5, 20)  # Zipf 分布模拟真实 token 频率
        ax2.bar(range(len(token_freq)), sorted(token_freq, reverse=True), color='lightcoral', alpha=0.7)
        ax2.set_title('Token Frequency Distribution')
        ax2.set_xlabel('Token Rank')
        ax2.set_ylabel('Frequency')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def demonstrate_batching(self):
        """演示 Batching 概念"""
        print("📦 概念 2: Batching (批处理)")
        print("=" * 50)
        print("批处理是将多个请求组合在一起同时处理，以提高效率")
        print()
        
        # 单个处理 vs 批处理对比
        requests = self.sample_texts
        
        print("🔄 单个处理模式:")
        single_start = time.time()
        for i, request in enumerate(requests, 1):
            processing_time = random.uniform(0.1, 0.3)  # 模拟处理时间
            time.sleep(processing_time)
            print(f"  请求 {i}: {request[:30]}... (耗时: {processing_time:.3f}s)")
        single_total = time.time() - single_start
        
        print(f"\n单个处理总时间: {single_total:.3f}s")
        
        print("\n📦 批处理模式:")
        batch_start = time.time()
        batch_processing_time = max(random.uniform(0.1, 0.3) for _ in requests)  # 批处理时间
        time.sleep(batch_processing_time)
        print(f"  批处理 {len(requests)} 个请求 (耗时: {batch_processing_time:.3f}s)")
        batch_total = time.time() - batch_start
        
        print(f"\n批处理总时间: {batch_total:.3f}s")
        print(f"效率提升: {(single_total / batch_total):.1f}x")
        
        # 可视化批处理效果
        self.visualize_batching_effect(single_total, batch_total)
    
    def visualize_batching_effect(self, single_time: float, batch_time: float):
        """可视化批处理效果"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        
        # 处理时间对比
        methods = ['Single Processing', 'Batch Processing']
        times = [single_time, batch_time]
        colors = ['lightcoral', 'lightgreen']
        
        bars = ax1.bar(methods, times, color=colors, alpha=0.7)
        ax1.set_title('Processing Time Comparison')
        ax1.set_ylabel('Time (seconds)')
        ax1.grid(True, alpha=0.3)
        
        # 添加数值标签
        for bar, time_val in zip(bars, times):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{time_val:.3f}s', ha='center', va='bottom')
        
        # 批大小 vs 吞吐量关系 (模拟)
        batch_sizes = [1, 2, 4, 8, 16, 32]
        throughput = [1/single_time * bs * 0.8**(bs-1) for bs in batch_sizes]  # 模拟吞吐量
        
        ax2.plot(batch_sizes, throughput, 'o-', color='blue', linewidth=2, markersize=6)
        ax2.set_title('Batch Size vs Throughput')
        ax2.set_xlabel('Batch Size')
        ax2.set_ylabel('Throughput (requests/s)')
        ax2.grid(True, alpha=0.3)
        ax2.set_xscale('log', base=2)
        
        plt.tight_layout()
        plt.show()
    
    def demonstrate_sampling(self):
        """演示 Sampling 策略"""
        print("🎲 概念 3: Sampling (采样策略)")
        print("=" * 50)
        print("采样策略决定了模型如何从概率分布中选择下一个 token")
        print()
        
        # 模拟概率分布
        vocab_size = 10
        logits = np.random.randn(vocab_size)  # 模拟模型输出
        probabilities = np.exp(logits) / np.sum(np.exp(logits))  # Softmax
        
        tokens = [f"token_{i}" for i in range(vocab_size)]
        
        print("模型输出概率分布:")
        for token, prob in zip(tokens, probabilities):
            print(f"  {token}: {prob:.3f}")
        
        print("\n不同采样策略的结果:")
        
        # 1. Greedy Sampling
        greedy_choice = np.argmax(probabilities)
        print(f"1. Greedy Sampling: {tokens[greedy_choice]} (概率: {probabilities[greedy_choice]:.3f})")
        
        # 2. Random Sampling
        random_choice = np.random.choice(vocab_size, p=probabilities)
        print(f"2. Random Sampling: {tokens[random_choice]} (概率: {probabilities[random_choice]:.3f})")
        
        # 3. Top-k Sampling
        k = 3
        top_k_indices = np.argsort(probabilities)[-k:]
        top_k_probs = probabilities[top_k_indices]
        top_k_probs = top_k_probs / np.sum(top_k_probs)  # 重新归一化
        top_k_choice = np.random.choice(top_k_indices, p=top_k_probs)
        print(f"3. Top-{k} Sampling: {tokens[top_k_choice]} (概率: {probabilities[top_k_choice]:.3f})")
        
        # 可视化采样策略
        self.visualize_sampling_strategies(tokens, probabilities, k)
    
    def visualize_sampling_strategies(self, tokens: List[str], probabilities: np.ndarray, k: int):
        """可视化采样策略"""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 8))
        
        # 原始概率分布
        ax1.bar(range(len(tokens)), probabilities, color='skyblue', alpha=0.7)
        ax1.set_title('Original Probability Distribution')
        ax1.set_xlabel('Token Index')
        ax1.set_ylabel('Probability')
        ax1.grid(True, alpha=0.3)
        
        # Greedy Sampling
        greedy_probs = np.zeros_like(probabilities)
        greedy_probs[np.argmax(probabilities)] = 1.0
        ax2.bar(range(len(tokens)), greedy_probs, color='red', alpha=0.7)
        ax2.set_title('Greedy Sampling')
        ax2.set_xlabel('Token Index')
        ax2.set_ylabel('Selection Probability')
        ax2.grid(True, alpha=0.3)
        
        # Top-k Sampling
        top_k_probs = np.zeros_like(probabilities)
        top_k_indices = np.argsort(probabilities)[-k:]
        top_k_probs[top_k_indices] = probabilities[top_k_indices]
        top_k_probs = top_k_probs / np.sum(top_k_probs) if np.sum(top_k_probs) > 0 else top_k_probs
        ax3.bar(range(len(tokens)), top_k_probs, color='green', alpha=0.7)
        ax3.set_title(f'Top-{k} Sampling')
        ax3.set_xlabel('Token Index')
        ax3.set_ylabel('Selection Probability')
        ax3.grid(True, alpha=0.3)
        
        # Temperature 效果
        temperatures = [0.5, 1.0, 2.0]
        for temp in temperatures:
            temp_probs = np.exp(np.log(probabilities + 1e-10) / temp)
            temp_probs = temp_probs / np.sum(temp_probs)
            ax4.plot(range(len(tokens)), temp_probs, 'o-', label=f'T={temp}', linewidth=2)
        
        ax4.set_title('Temperature Effect on Sampling')
        ax4.set_xlabel('Token Index')
        ax4.set_ylabel('Probability')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def run_all_demonstrations(self):
        """运行所有概念演示"""
        print("🧠 nano-vLLM 核心概念深度理解")
        print("=" * 60)
        print("通过可视化和实验理解关键技术概念\n")
        
        self.demonstrate_tokenization()
        print("\n" + "-" * 60 + "\n")
        
        self.demonstrate_batching()
        print("\n" + "-" * 60 + "\n")
        
        self.demonstrate_sampling()
        
        print("\n" + "=" * 60)
        print("🎓 概念学习总结")
        print("=" * 60)
        print("1. Tokenization: 文本 → 数字序列的转换过程")
        print("2. Batching: 提高处理效率的关键技术")
        print("3. Sampling: 控制生成文本多样性的策略")
        print("\n💡 这些概念是理解 vLLM 工作原理的基础！")

# 运行概念演示
demonstrator = ConceptDemonstrator()
demonstrator.run_all_demonstrations()

## 🚀 下一步学习

恭喜您完成了 nano-vLLM 的环境设置和快速开始！

### 📚 推荐学习路径

1. **基础概念深入** → [01_基础推理和批处理.ipynb](01_基础推理和批处理.ipynb)
2. **核心技术理解** → [02_PagedAttention和内存管理.ipynb](02_PagedAttention和内存管理.ipynb)
3. **高级特性** → [03_智能调度和性能优化.ipynb](03_智能调度和性能优化.ipynb)
4. **实战项目** → [04_端到端推理服务.ipynb](04_端到端推理服务.ipynb)

### 💡 学习建议

- 🔬 **动手实验**: 修改代码参数，观察结果变化
- 📊 **性能对比**: 比较不同配置的性能差异
- 🤔 **深入思考**: 理解每个技术的设计原理
- 📝 **记录笔记**: 记录重要概念和实验结果

### 🔗 相关资源

- [项目文档](https://github.com/zysno1/nano-vllm-learning/tree/main/docs)
- [常见问题](https://github.com/zysno1/nano-vllm-learning/blob/main/docs/faq.md)
- [实践任务](https://github.com/zysno1/nano-vllm-learning/blob/main/docs/practice_tasks.md)